In [1]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

import datetime

import pandas as pd

from pandas import ExcelWriter

from selenium import webdriver

import math

import time

from time import sleep

import os

from selenium.webdriver.common.by import By

from bs4 import BeautifulSoup

from selenium.webdriver.common.keys import Keys

from selenium.webdriver.support.ui import Select

from selenium.webdriver.support import expected_conditions

from webdriver_manager.chrome import ChromeDriverManager

# %%

In [2]:
# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'CR SUPEN' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.0")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - moodys.com\\Desktop\\Regulator\\{regulatorName}"
#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename, engine='openpyxl')

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process


if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)

Running CR SUPEN Web Scraping Tool v.1.0


In [3]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()

# %%

In [4]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict={

         'CR SUPEN 1': 'https://www.supen.fi.cr/entidades-supervisadas',

        }



Typology={

        'CR SUPEN 1':    'Entidades Supervisadas',

        }



sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [],
        }


now = datetime.datetime.now()

processdate = now.strftime('%Y-%m-%d')




In [5]:
# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict


In [6]:

# %%

#------------------------------------------------ Begin_Main ----------------------------------------

for k, reg in enumerate(regdict):

    print(f"[INFO] : Working {k+1}/{len(regdict)} _({reg})_ ")

    driver.get(regdict[reg])   
    
    sleep(2)
    
    soup=BeautifulSoup(driver.page_source, 'html.parser')
    
    sleep(2)
    
    data = soup.find_all(class_='card-body')
    for d in data:
        title = d.find('span').text.strip()
        infos = d.find('ul').find_all('li')
        #print(len(infos))
        if len(infos)>=3:
            print(title)
            sqldict['Name'].append(title)
            sqldict['ListProcessDate'].append(processdate)
            for i,info in enumerate(infos):
                index = info.text.strip().find(':')
                if i==0:
                    sqldict['Website'].append(info.text[index+2:].strip())
                    # print(info.text[:index+1].strip())
                    # print(info.text[index+2:].strip())
                if i==1:
                    sqldict['Email'].append(info.text[index+2:].strip())
                    # print(info.text[:index+1].strip())
                    # print(info.text[index+2:].strip())
                    pass
                if i ==2:
                    phone = ''
                    if len(info.text[index+2:].strip())>20:
                        print(info.text[index+2:].strip())
                        if '(506)'in info.text[index+2:].strip():
                            phone = info.text[index+2:].strip()[:15].strip()
                        else:
                            phone = info.text[index+2:].strip()[:10].strip()
                    else:
                        phone = info.text[index+2:].strip()
                    print(phone)
                    sqldict['Phone'].append(phone)
                    # print(info.text[:index+1].strip())
                    # print(info.text[index+2:].strip())
                if i==3 and 'Direc' in info.text[:index+1]:
                    # print(info.text[:index+1].strip())
                    # print(info.text[index+2:].strip())
                    sqldict['Address_1'].append(info.text[index+2:].strip())
            sqldict['RegCtry'].append(reg.split(' ')[0]) 
            sqldict['RegCode'].append(reg.split(' ')[1])
            sqldict['ListCode'].append(reg.split(' ')[-1])
            sqldict['ListName'].append(Typology[reg])
            sqldict['RegulationType'].append('Regulated')   
            sqldict = bourange_same_length_array(sqldict)   
        

[INFO] : Working 1/1 _(CR SUPEN 1)_ 
BN Vital
(506) 2212-0900
BCR Pensiones
(506) 2211-1111 opción 3
(506) 2211-1111
Vida Plena
(506) 2523-5200
OPC CCSS
(506) 2522-3600
BAC Pensiones
(506) 2295-9200
Popular Pensiones
(506) 2010-0300
Régimen de Invalidez, Vejez y Muerte (CCSS)
2284-9200 extensiones 91081280 (secretaria de la jefatura de la subárea de Trámite de Pensiones), 91081034 (Subárea de Plataforma de Servicios)
2284-9200
Junta de Pensiones del Magisterio Nacional
2284-6500
Fondo de Jubilaciones y Pensiones del Poder Judicial
2549-1594 ó 2549-1519
2549-1594
Fondo de Pensiones del Benemérito Cuerpo de Bomberos
(506) 2547-3700
Dirección Nacional de Pensiones
2542 0000


In [8]:
# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------


os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df  = df.drop_duplicates()
df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

sleep(3)

C:\Users\wuj1\AppData\Local\Temp\2\ipykernel_13952\350249288.py:11: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [9]:
df.to_csv('ver2.csv')